# Parte 1 · Corpus y extracción (OCR + ASR)

**Pregunta guía:** ¿podemos convertir documentos escaneados y entrevistas en audio peruanos en un
archivo de texto estructurado y buscable?

| | |
|---|---|
| **Fecha de extracción** | Audios: 22/10/2026 · Documentos de Commons: 22/10/2026 · Capturas propias: ver `datos/fuentes.csv` |
| **Fuentes** | Wikimedia Commons (entrevistas peruanas y documentos), OpenSLR 73 (español peruano leído, Google), capturas propias sin datos personales |
| **Licencias** | CC BY-SA 4.0 (todos los audios), y en documentos: dominio público, CC0, CC BY 2.0, CC BY 4.0 y CC BY-SA 4.0. El detalle por archivo está en `datos/fuentes.csv` y en la celda de la sección 1 |
| **OCR** | PaddleOCR 3.7.0, `lang='es'` (carga los modelos PP-OCRv6_medium_det y PP-OCRv6_medium_rec) |
| **ASR** | faster-whisper 1.2.1 con el modelo **small** (CPU, int8), VAD Silero incluido en faster-whisper |
| **Diarización** | pyannote.audio 4.0.7 con WeSpeaker ResNet34 + agrupamiento jerárquico (`voz.py` de la Sesión 13) |
| **Entorno** | `uv sync --group docai --group audio --group tarea2` (grupos `docai` y `audio` del curso + lo propio de la tarea) |

**Cómo se lee este notebook.** El trabajo pesado (OCR y transcripción) vive en `src/` y guarda su avance en
`datos/raw/` (fuera de Git). Las celdas llaman a esas mismas funciones: si un documento o clip ya se procesó,
se reutiliza; si falta, se procesa. Así el notebook muestra el pipeline real sin repetir una hora de cómputo.

> Nota técnica (Windows): `torch` debe importarse antes que `paddle` (si no, `WinError 127` en `shm.dll`), y
> PaddlePaddle 3.3 falla en CPU con oneDNN, por lo que se usa `enable_mkldnn=False`. Ninguna de las dos cosas
> cambia el modelo ni sus resultados; solo permiten que corra.

In [ ]:
import torch  # primero, por el problema de DLL de Windows con paddle
import sys, warnings, hashlib, json, re
from datetime import datetime
from importlib.metadata import version
from pathlib import Path

import pandas as pd
import plotly.express as px
import plotly.io as pio

warnings.filterwarnings("ignore")
sys.path.insert(0, "src")
RAW = Path("datos/raw")
pd.set_option("display.max_colwidth", 80)

print("Ejecutado:", datetime.now().strftime("%d/%m/%Y %H:%M"))
import transformers
for p in ["paddleocr", "paddlepaddle", "faster-whisper", "pyannote.audio", "torch", "jiwer", "plotly"]:
    print(f"  {p:15s} {version(p)}")
print(f"  {'transformers':15s} {transformers.__version__}")

## 1. Fuentes y licencias

Cada archivo tiene su URL, licencia y forma de recuperarlo en `datos/fuentes.csv`. Las licencias se leyeron con la
API de Commons (campo `LicenseShortName`) y en la página oficial de OpenSLR 73. Para no repetir nombres de
personas, la atribución CC BY-SA se hace con el enlace a la página de Commons (licencia, sección 3(a)(2)).

In [ ]:
fuentes = pd.read_csv("datos/fuentes.csv")
fuentes.groupby(["modalidad", "origen", "licencia"]).size().rename("archivos").to_frame()

## 2. OCR con PaddleOCR

**Parámetros (fijados antes de ver resultados, `modelos/config_evaluacion.json`):** `lang='es'`, sin clasificar la
orientación de la página, sin "planchar" hojas dobladas y sin girar líneas; **sin preprocesamiento** de la imagen.
Es la configuración de la Sesión 10 y también la línea base de la Parte 2: no se tunea con los documentos.

Respaldo crudo por documento en `datos/raw/documentos/ocr/` (JSON con cajas, textos y confianzas, e imagen con las
cajas dibujadas) y un recorte por línea detectada en `datos/raw/documentos/recortes/<id>/`.

In [ ]:
import ocr_documentos
ocr_documentos.main()   # reutiliza los documentos ya procesados; procesa los que falten
ocr = pd.read_csv(RAW / "documentos" / "ocr_crudo.csv")
ocr["caracteres_ocr"] = ocr["texto_ocr_crudo"].fillna("").str.len()
ocr.groupby("tipo").agg(documentos=("id", "size"), lineas_mediana=("n_lineas", "median"),
                        confianza_media=("confianza_media", "mean"), segundos_por_doc=("segundos_ocr", "median")).round(2)

La **confianza** es la que el propio modelo se asigna: no es exactitud. La exactitud (CER) se mide en la Parte 2
contra transcripciones manuales.

## 3. Transcripción (ASR): VAD, faster-whisper y diarización

1. **VAD** (Silero): solo se le pasa voz al modelo, así no "inventa" frases en silencios.
2. **faster-whisper small**, español, `beam_size=5`.
3. **Diarización**: huella de voz (WeSpeaker) por segmento y agrupamiento; el número de hablantes no se fija.
4. **Exportación** por clip a CSV (segmentos con tiempos y hablante), **SRT** (subtítulos) y **TXT** (por turnos),
   en `datos/raw/audio/salidas/` (fuera de Git: antes de seudonimizar).

In [ ]:
import procesar_audio
procesar_audio.main()   # reutiliza los clips ya transcritos
asr = pd.read_csv(RAW / "audio" / "asr_crudo.csv")
salidas = sorted((RAW / "audio" / "salidas").glob("clip_*.*"))
print("archivos exportados:", pd.Series([p.suffix for p in salidas]).value_counts().to_dict())

## 4. Limpieza: duplicados, audios sin voz, documentos ilegibles

- **Duplicados exactos:** mismo hash SHA-256 del archivo.
- **Misma grabación en otro formato** (audio): el hash no la detecta porque los bytes cambian; se compara la
  envolvente de energía (correlación ≥ 0,95 con duración casi igual).
- **Audio sin voz:** menos de 10 % del audio con voz según el VAD.
- **Documento ilegible:** que una persona no pueda leerlo (revisión visual de cada imagen antes del OCR). Que el
  OCR falle en un manuscrito legible no lo vuelve "ilegible": es justamente lo que hay que medir.

In [ ]:
docs = pd.read_csv(RAW / "documentos" / "corpus_documentos.csv")
h = docs["archivo_original"].map(lambda p: hashlib.sha256(Path(p).read_bytes()).hexdigest())
print("Documentos: duplicados exactos por hash =", int(h.duplicated().sum()),
      "| ilegibles en la revisión visual = 0 (de", len(docs), ")")
lim = pd.read_csv(RAW / "audio" / "limpieza.csv")
print("Audios descargados:", len(lim), "| descartados:", int(lim["descartado"].sum()))
lim.loc[lim["descartado"], ["id", "motivo_descarte", "duracion_seg"]]

### Revisión de posibles alucinaciones de Whisper

Indicadores por clip (no reemplazan escuchar; señalan dónde mirar): palabras por minuto (muy alto = posible texto
inventado; muy bajo = voz perdida), trigramas repetidos (bucles típicos de Whisper) y frases típicas de alucinación.

In [ ]:
es = asr[(asr["idioma"] == "es") & ~asr["descartado"]].copy()
es["palabras"] = es["texto_plano"].fillna("").str.split().str.len()
es["palabras_por_min"] = (es["palabras"] / (es["duracion_seg"] / 60)).round(0)
def max_trigrama(t):
    w = re.findall(r"\w+", str(t).lower()); tri = [" ".join(w[i:i + 3]) for i in range(len(w) - 2)]
    return max([tri.count(g) for g in set(tri)] or [0])
es["max_trigrama_repetido"] = es["texto_plano"].map(max_trigrama)
cols = ["id", "condicion", "duracion_seg", "proporcion_voz", "palabras_por_min", "max_trigrama_repetido", "n_hablantes"]
es.loc[(es["palabras_por_min"] > 170) | (es["palabras_por_min"] < 40) | (es["max_trigrama_repetido"] >= 5)
       | es["sospechas_alucinacion"].notna(), cols]

**Lectura de los casos señalados** (revisados segmento por segmento el 23/10):

- **clip_015** (marcha, 186 palabras/min): el texto es coherente; la frase marcada ("ya, ya, ya, ya") es habla real.
  El problema no es el ASR sino la **diarización**: los segmentos de ~1 s dan huellas de voz poco fiables y salen
  15 "hablantes" en un intercambio que, según el texto, es de preguntas y respuestas entre quien entrevista y
  quien responde (el número real de voces se anota al escuchar el clip en la Parte 2).
- **clip_010** (fiesta patronal, 18 palabras/min): el caso inverso, **voz perdida**. Solo 11 % del audio pasa el
  VAD (probablemente por la música de la fiesta) y se transcriben los primeros 14 s de 98. Se mantiene en el corpus:
  es un caso de "audio con música" para la Parte 2.
- **clip_002** (trigrama repetido 5 veces): repetición real del entrevistador ("sigo con la siguiente pregunta").

**Hallazgo:** con el número de hablantes sin fijar, la diarización **sobreestima** las voces (hasta 15 en un clip).
No se corrige el umbral aquí porque quedó fijado antes de ver resultados; se evalúa en la Parte 2 contra el
número real de voces anotado a mano.

## 5. Seudonimización

Tres capas, de la más segura a la más amplia: (1) lista de nombres conocidos (títulos y descripciones de Commons),
con comparación difusa para atrapar nombres mal escritos por el OCR/ASR; (2) NER de personas
(`mrm8488/bert-spanish-cased-finetuned-ner`, umbral bajo a propósito); (3) expresiones regulares para DNI, RUC,
celulares y correos. Cada persona recibe un seudónimo estable `[PERSONA_###]`. Se seudonimizan **todas** las
personas, también las históricas o públicas (criterio conservador). La tabla nombre → seudónimo y la lista de
detecciones quedan en `datos/raw/` (fuera de Git), para la **revisión humana**, que el procedimiento automático no
reemplaza.

In [ ]:
import seudonimizar
seudonimizar.main()
ocr_s = pd.read_csv(RAW / "documentos" / "ocr_seudonimizado.csv")
asr_s = pd.read_csv(RAW / "audio" / "asr_seudonimizado.csv")
conteo = pd.DataFrame({
    m: d[["n_reemplazos_lista", "n_reemplazos_ner", "n_reemplazos_regex"]].sum()
    for m, d in [("documentos", ocr_s), ("audio", asr_s)]}).T
conteo["total"] = conteo.sum(axis=1)
print("Cadenas distintas reemplazadas por un seudónimo:", len(pd.read_csv(RAW / "seudonimos.csv")),
      "(una misma persona escrita de dos formas cuenta dos veces)")
conteo

## 6. Análisis exploratorio

Paleta propia: verde selva y morado sobre fondo crema. El verde de los datos (`#0F7A55`) es una versión más
saturada del verde de títulos (`#1F5F4A`), que como color de barra se leía gris; ambos colores de datos pasan el
validador de paleta (luminosidad, croma, separación para daltonismo y contraste sobre el fondo).
Cada gráfico se guarda además como imagen para que se vea en GitHub.

In [ ]:
CREMA, CARBON, VERDE_TITULO, VERDE, MORADO = "#F7F3EA", "#2B2B2B", "#1F5F4A", "#0F7A55", "#6A4C93"
pio.templates["propia"] = pio.templates["plotly_white"]
pio.templates["propia"].layout.update(
    paper_bgcolor=CREMA, plot_bgcolor=CREMA, font=dict(family="Segoe UI, Arial", color=CARBON, size=13),
    title=dict(font=dict(color=VERDE_TITULO, size=17), x=0.01, xanchor="left"),
    xaxis=dict(gridcolor="#E4DED0", zeroline=False), yaxis=dict(gridcolor="#E4DED0", zeroline=False),
    margin=dict(l=70, r=30, t=90, b=60))
pio.templates.default = "propia"
pio.renderers.default = "plotly_mimetype+png"   # interactivo en Jupyter + imagen visible en GitHub
FIG = Path("figuras"); FIG.mkdir(exist_ok=True)
def mostrar(fig, nombre, alto=480):
    fig.update_layout(height=alto, width=980)
    fig.write_html(FIG / f"{nombre}.html", include_plotlyjs="cdn")
    fig.show()

In [ ]:
# Títulos: cada uno dice la conclusión; las cifras salen de los datos (el sábado se agregan capturas propias)
MINIMO = {"impreso": 8, "foto": 8, "manuscrito": 9}
cuenta = ocr["tipo"].value_counts()
faltan = {t: MINIMO[t] - cuenta.get(t, 0) for t in MINIMO if cuenta.get(t, 0) < MINIMO[t]}
if faltan:
    TITULO_1 = ("Con licencia libre sobran impresos peruanos, no manuscritos: faltan "
                + " y ".join(f"{n} {t}{'s' if n > 1 else ''}" for t, n in faltan.items()) + " para el mínimo")
else:
    TITULO_1 = (f"Corpus completo ({len(ocr)} documentos): los manuscritos, escasos con licencia libre, "
                "se completaron con capturas propias")

ETIQUETA = {"evento_publico": "eventos públicos", "calle": "la calle", "marcha": "marchas",
            "fiesta_patronal": "fiestas patronales", "leido_estudio": "lectura en estudio"}
miles = lambda x: f"{x:,.0f}".replace(",", ".")
leidos = es[es["origen"] == "OpenSLR 73"]
top = es.groupby("condicion")["duracion_seg"].agg(["size", "sum"]).sort_values("sum", ascending=False).iloc[0]
TITULO_2 = (f"{len(leidos)} de {len(es)} clips son frases leídas de ~{leidos['duracion_seg'].median():.0f} s; "
            f"{int(top['size'])} entrevistas en {ETIQUETA[top.name]} concentran el "
            f"{top['sum'] / es['duracion_seg'].sum():.0%} del audio")

med = ocr.groupby("tipo")["caracteres_ocr"].median()
TITULO_3 = (f"Los manuscritos rinden menos texto OCR (mediana {miles(med['manuscrito'])} caracteres vs. "
            f"{miles(med.drop('manuscrito').min())}-{miles(med.drop('manuscrito').max())}) y {(es['palabras'] <= 20).sum()} "
            f"de {len(es)} clips tienen 20 palabras o menos")
print(TITULO_1, TITULO_2, TITULO_3, sep="\n")

In [ ]:
# Gráfico 1: documentos por tipo
n_tipo = (ocr["tipo"].value_counts().reindex(["impreso", "foto", "manuscrito"]).fillna(0).astype(int)
          .rename_axis("tipo").reset_index(name="documentos"))
fig = px.bar(n_tipo, x="tipo", y="documentos", text="documentos", color_discrete_sequence=[VERDE],
             title=TITULO_1)
fig.update_traces(textposition="outside", marker_line_width=0, width=0.55,
                  hovertemplate="%{x}: %{y} documentos<extra></extra>")
# mínimo de cada tipo: una marca corta sobre su propia barra (no una línea que cruce todo el gráfico)
fig.add_scatter(x=list(MINIMO), y=list(MINIMO.values()), mode="markers+text", name="mínimo exigido",
                marker=dict(symbol="line-ew", size=60, line=dict(width=3, color=MORADO)),
                text=[f"mínimo {v}" for v in MINIMO.values()], textposition="top center",
                textfont=dict(color=CARBON), hovertemplate="mínimo exigido: %{y}<extra></extra>")
fig.update_layout(showlegend=False)
fig.update_yaxes(title="documentos", range=[0, max(11, n_tipo["documentos"].max() + 2)])
fig.update_xaxes(title=None)
mostrar(fig, "g1_documentos_por_tipo")

In [ ]:
# Gráfico 2: duración por clip, coloreada por fuente
dur = es.sort_values("duracion_seg", ascending=True).copy()
dur["fuente"] = dur["origen"].map({"Wikimedia Commons": "Entrevista (Commons)", "OpenSLR 73": "Frase leída (OpenSLR 73)"})
fig = px.bar(dur, x="duracion_seg", y="id", color="fuente", orientation="h",
             color_discrete_map={"Entrevista (Commons)": VERDE, "Frase leída (OpenSLR 73)": MORADO},
             hover_data={"condicion": True, "duracion_seg": ":.1f", "id": False},
             labels={"duracion_seg": "duración (segundos)", "id": "", "condicion": "condición"},
             title=TITULO_2)
fig.update_traces(marker_line_color=CREMA, marker_line_width=1)
fig.update_layout(legend=dict(title=None, orientation="h", y=1.06, x=0), bargap=0.25)
fig.update_yaxes(tickfont=dict(size=10))
mostrar(fig, "g2_duracion_por_clip", alto=760)

In [ ]:
# Gráfico 3: longitud de los textos (dos paneles, cada uno con su eje: caracteres y palabras no se mezclan)
from plotly.subplots import make_subplots
import plotly.graph_objects as go
fig = make_subplots(rows=1, cols=2, subplot_titles=("Documentos: caracteres leídos por el OCR",
                                                    "Clips: palabras transcritas por el ASR"), horizontal_spacing=0.1)
fig.add_trace(go.Histogram(x=ocr["caracteres_ocr"], nbinsx=12, marker_color=VERDE, marker_line_color=CREMA,
                           marker_line_width=2, hovertemplate="%{x} caracteres: %{y} documentos<extra></extra>"), 1, 1)
fig.add_trace(go.Histogram(x=es["palabras"], nbinsx=20, marker_color=MORADO, marker_line_color=CREMA,
                           marker_line_width=2, hovertemplate="%{x} palabras: %{y} clips<extra></extra>"), 1, 2)
fig.update_xaxes(title_text="caracteres por documento", row=1, col=1)
fig.update_xaxes(title_text="palabras por clip", row=1, col=2)
fig.update_yaxes(title_text="documentos", dtick=1, row=1, col=1)
fig.update_yaxes(title_text="clips", row=1, col=2)
fig.update_layout(title=TITULO_3, showlegend=False, bargap=0.05)
mostrar(fig, "g3_longitud_textos")

## 7. Validación: los extremos tienen que cuadrar con los histogramas

Se muestran los 3 documentos con más y menos texto OCR y los 3 clips más largos y más cortos. El texto que se ve
ya está **seudonimizado**.

In [ ]:
v_doc = ocr.merge(ocr_s[["id", "texto_ocr"]], on="id")[["id", "tipo", "caracteres_ocr", "n_lineas", "texto_ocr"]]
v_doc["inicio_del_texto"] = v_doc["texto_ocr"].fillna("").str.replace("\n", " ").str[:90]
print("Documentos con MÁS texto OCR")
display(v_doc.nlargest(3, "caracteres_ocr").drop(columns="texto_ocr"))
print("Documentos con MENOS texto OCR")
display(v_doc.nsmallest(3, "caracteres_ocr").drop(columns="texto_ocr"))

In [ ]:
v_clip = es.merge(asr_s[["id", "texto_plano_seud"]], on="id")[["id", "condicion", "duracion_seg", "palabras", "texto_plano_seud"]]
v_clip["inicio_del_texto"] = v_clip["texto_plano_seud"].fillna("").str[:90]
print("Clips MÁS largos")
display(v_clip.nlargest(3, "duracion_seg").drop(columns="texto_plano_seud"))
print("Clips MÁS cortos")
display(v_clip.nsmallest(3, "duracion_seg").drop(columns="texto_plano_seud"))

VALIDACION_TEXTO

## 8. Exportación

Solo texto **seudonimizado**. `texto_referencia` queda vacío por ahora: se llena en la Parte 2 únicamente para la
muestra de evaluación.

In [ ]:
fuentes_idx = fuentes.set_index("id")
documentos = ocr.drop(columns=["texto_ocr_crudo"]).merge(ocr_s[["id", "texto_ocr"]], on="id").assign(
    archivo=lambda d: d["archivo_original"],
    fuente=lambda d: d["id"].map(lambda i: f"{fuentes_idx.loc[i, 'origen']} ({fuentes_idx.loc[i, 'licencia']})"),
    modelo_ocr=lambda d: d["modelo_ocr"], texto_referencia="")
documentos = documentos[["id", "archivo", "tipo", "fuente", "texto_ocr", "modelo_ocr", "texto_referencia"]]

transcripciones = es.drop(columns=["texto_asr", "texto_plano"]).merge(asr_s[["id", "texto_asr_seud"]], on="id").assign(
    idioma="es (español peruano, según la fuente)", texto_referencia="")
transcripciones = transcripciones.rename(columns={"texto_asr_seud": "texto_asr"})[
    ["id", "archivo_audio", "duracion_seg", "idioma", "texto_asr", "modelo_asr", "n_hablantes", "texto_referencia"]]

documentos.to_csv("datos/documentos.csv", index=False)
transcripciones.to_csv("datos/transcripciones.csv", index=False)
fuentes.to_csv("datos/fuentes.csv", index=False)
assert list(transcripciones.columns).count("texto_asr") == 1 and "texto_ocr_crudo" not in documentos
for nombre, df in [("documentos", documentos), ("transcripciones", transcripciones), ("fuentes", fuentes)]:
    print(f"datos/{nombre}.csv  shape = {df.shape}")

In [ ]:
documentos.head(5)

In [ ]:
transcripciones.head(5)

In [ ]:
fuentes[["id", "modalidad", "fuente_url", "licencia", "acceso"]].head(5)